# Qwen3-VL: Zero-Shot Object Detection, Action Recognition & Scene Understanding

## Research-Grade Vision-Language Evaluation

This notebook evaluates **Qwen3-VL-2B-Instruct** as a zero-shot vision-language perception model for structured image understanding.

### Research objectives
- Detect visible objects using open-ended visual reasoning.
- Infer observable object actions/states when sufficient visual evidence exists.
- Estimate 2D bounding boxes in the original image coordinate system.
- Generate a concise scene-level description.
- Enforce a machine-readable JSON schema.
- Validate predicted bounding boxes and confidence values automatically.
- Record latency, memory usage, configuration, and raw/structured outputs for reproducibility.

### Research position

This is a **VLM reasoning baseline**, not a replacement for a dedicated object detector. VLM-generated bounding boxes should not be treated as detector-grade localization without quantitative evaluation against annotated ground truth.

For a publication-quality comparison, the same image set, class definitions, prompts, and evaluation metrics should later be used across Qwen3-VL, Qwen2.5-VL, YOLOE, OWLv2, GroundingDINO, and relevant video/action baselines.


## 1. Environment & Reproducibility


In [ ]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q -U bitsandbytes accelerate huggingface_hub pillow matplotlib
# RESTART THE RUNTIME

In [ ]:
import os
import gc
import json
import re
import time
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from pathlib import Path
from IPython.display import display
from huggingface_hub import hf_hub_download

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Input Image


In [ ]:
ASSET_REPO = "WasiqSaleem/Portfolio_Assest"
ASSET_FILE = "cat.jpg"
IMAGE_PATH = Path("/content/cat.jpg")

cached_image = hf_hub_download(
    repo_id=ASSET_REPO,
    filename=ASSET_FILE,
    repo_type="dataset",
)

if not IMAGE_PATH.exists():
    import shutil
    shutil.copy2(cached_image, IMAGE_PATH)

image = Image.open(IMAGE_PATH).convert("RGB")
IMAGE_WIDTH, IMAGE_HEIGHT = image.size

print(f"Image: {IMAGE_PATH}")
print(f"Resolution: {IMAGE_WIDTH} × {IMAGE_HEIGHT} px")
display(image)


## 3. Memory & Runtime Utilities


In [ ]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def memory_report(label=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(
            f"[{label}] "
            f"allocated={allocated:.2f} GB | "
            f"reserved={reserved:.2f} GB | "
            f"peak={peak:.2f} GB"
        )
    else:
        print(f"[{label}] CUDA unavailable")


## 4. Research Prompt & Output Schema

The prompt is deliberately strict. The model must distinguish **visible evidence** from unsupported inference.

Bounding boxes use the original image pixel coordinate system:

`[x1, y1, x2, y2]`

where `(0, 0)` is the top-left corner.


In [ ]:
SYSTEM_PROMPT = f"""
You are a research-grade visual perception system performing zero-shot image understanding.

Analyze the supplied RGB image and return ONLY ONE valid JSON object.
Do not output markdown, comments, explanations, or any text outside JSON.

Required schema:

{{
  "objects": [
    {{
      "label": "short concrete object name",
      "bbox_2d": [x1, y1, x2, y2],
      "action": "observable action or state",
      "confidence": 0.0
    }}
  ],
  "scene_description": "concise evidence-based scene description"
}}

IMAGE GEOMETRY:
- Image width = {IMAGE_WIDTH} pixels.
- Image height = {IMAGE_HEIGHT} pixels.
- x increases from left to right.
- y increases from top to bottom.
- bbox_2d must contain four integer pixel coordinates.
- Coordinates must satisfy:
  0 <= x1 < x2 <= {IMAGE_WIDTH}
  0 <= y1 < y2 <= {IMAGE_HEIGHT}

EVIDENCE RULES:
1. Report only objects visibly supported by the image.
2. Do not invent hidden objects.
3. Describe only observable actions or states.
4. If an object's action cannot be established, use "unknown".
5. Do not infer unsupported intentions.
6. Confidence must be a number between 0 and 1.
7. Return objects=[] if no sufficiently visible object can be identified.
8. Keep labels short and concrete.
9. Keep scene_description concise and evidence-based.
"""

USER_PROMPT = """
Perform zero-shot object detection, observable action/state recognition,
and scene understanding on this image.

Return only the required JSON object.
"""


## 5. Load Qwen3-VL with 4-bit NF4 Quantization

4-bit NF4 is used to make the experiment practical on constrained GPU environments. The configuration is explicitly recorded for reproducibility.

In [ ]:
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_storage=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading: {MODEL_ID}")
load_start = time.perf_counter()

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    quantization_config=bnb_config,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

load_time = time.perf_counter() - load_start

print(f"Model loaded in {load_time:.2f} s")
memory_report("after model loading")


## 6. Zero-Shot Multimodal Inference


In [ ]:
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT}],
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": USER_PROMPT},
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

model_device = next(model.parameters()).device
inputs = {
    k: v.to(model_device) if hasattr(v, "to") else v
    for k, v in inputs.items()
}

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

print("Running Qwen3-VL inference...")

start = time.perf_counter()

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

inference_time = time.perf_counter() - start

peak_vram = (
    torch.cuda.max_memory_allocated() / 1e9
    if torch.cuda.is_available()
    else None
)

generated_ids_trimmed = generated_ids[:, inputs["input_ids"].shape[-1]:]

raw_output = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0].strip()

print(f"Inference time: {inference_time:.3f} s")

if peak_vram is not None:
    print(f"Peak VRAM: {peak_vram:.2f} GB")

print("\nRaw Model Output:\n")
print(raw_output)


## 7. Robust JSON Parsing & Structural Validation

A syntactically correct JSON response is not sufficient. The notebook also checks the schema and image-coordinate geometry.

In [ ]:
def parse_json_output(text):
    text = text.strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)

        if match:
            return json.loads(match.group(0))

        raise ValueError(
            "No valid JSON object found in model output."
        )


def validate_result(result, width, height):
    errors = []

    if not isinstance(result, dict):
        return ["Top-level output is not a JSON object."]

    if not isinstance(result.get("objects"), list):
        errors.append("'objects' must be a list.")

    if not isinstance(result.get("scene_description"), str):
        errors.append("'scene_description' must be a string.")

    if not isinstance(result.get("objects"), list):
        return errors

    for i, obj in enumerate(result["objects"]):

        if not isinstance(obj, dict):
            errors.append(f"Object {i}: must be a JSON object.")
            continue

        for key in ["label", "bbox_2d", "action", "confidence"]:
            if key not in obj:
                errors.append(f"Object {i}: missing '{key}'.")

        box = obj.get("bbox_2d")

        if isinstance(box, list) and len(box) == 4:
            x1, y1, x2, y2 = box

            if not all(isinstance(v, int) for v in box):
                errors.append(
                    f"Object {i}: bbox coordinates must be integers."
                )

            elif not (
                0 <= x1 < x2 <= width
                and 0 <= y1 < y2 <= height
            ):
                errors.append(
                    f"Object {i}: bbox is outside image bounds or invalid."
                )
        else:
            errors.append(
                f"Object {i}: bbox_2d must contain four coordinates."
            )

        confidence = obj.get("confidence")

        if not isinstance(confidence, (int, float)):
            errors.append(
                f"Object {i}: confidence must be numeric."
            )
        elif not 0 <= confidence <= 1:
            errors.append(
                f"Object {i}: confidence must be in [0, 1]."
            )

    return errors


try:
    parsed_result = parse_json_output(raw_output)
    validation_errors = validate_result(
        parsed_result,
        IMAGE_WIDTH,
        IMAGE_HEIGHT
    )

except Exception as exc:
    parsed_result = None
    validation_errors = [str(exc)]

print(
    "Validation status:",
    "PASS" if not validation_errors else "FAIL"
)

if validation_errors:
    for error in validation_errors:
        print("-", error)
else:
    print(json.dumps(parsed_result, indent=2, ensure_ascii=False))


## 8. Visualize VLM-Generated Bounding Boxes


In [ ]:
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)
ax.set_axis_off()
ax.set_title("Qwen3-VL Zero-Shot Object & Action Predictions")

if parsed_result and not validation_errors:

    for obj in parsed_result["objects"]:

        x1, y1, x2, y2 = obj["bbox_2d"]

        rect = Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=2
        )

        ax.add_patch(rect)

        label = (
            f"{obj['label']} | "
            f"{obj['action']} | "
            f"{obj['confidence']:.2f}"
        )

        ax.text(
            x1,
            max(0, y1 - 5),
            label,
            fontsize=9,
            backgroundcolor="white"
        )

plt.tight_layout()
plt.show()


## 9. Reproducible Research Record

The result is saved as a structured JSON artifact containing the experimental configuration, timing, memory footprint, raw VLM response, validated output, and validation errors.

In [ ]:
research_record = {
    "experiment": "Qwen3-VL zero-shot visual perception",
    "model": MODEL_ID,
    "quantization": {
        "method": "bitsandbytes",
        "bits": 4,
        "type": "NF4",
        "compute_dtype": "float16",
        "double_quantization": True,
    },
    "image": str(IMAGE_PATH),
    "image_resolution": [IMAGE_WIDTH, IMAGE_HEIGHT],
    "seed": SEED,
    "inference": {
        "latency_seconds": round(inference_time, 4),
        "peak_vram_gb": (
            round(peak_vram, 4)
            if peak_vram is not None
            else None
        ),
        "generation": {
            "max_new_tokens": 512,
            "sampling": False,
        },
    },
    "validation": {
        "status": "PASS" if not validation_errors else "FAIL",
        "errors": validation_errors,
    },
    "raw_output": raw_output,
    "structured_output": parsed_result,
}

RESULT_PATH = Path("/content/qwen3_vl_research_result.json")

with open(RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        research_record,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Research record saved to: {RESULT_PATH}")


## 10. Research Interpretation & Limitations

This experiment evaluates **zero-shot multimodal perception and reasoning**.

A syntactically correct JSON response does **not** establish that the predicted objects, actions, or bounding boxes are correct.

For the main research evaluation, use annotated images/videos and report:

### Object detection
- Precision
- Recall
- IoU
- mAP@0.5
- mAP@0.5:0.95

### Localization
- Mean/median IoU
- Percentage of predictions above IoU thresholds
- Center-point error

### Action recognition
- Per-class precision
- Recall
- F1-score
- Confusion matrix

### Efficiency
- End-to-end latency
- Peak VRAM
- Throughput
- Quantization level

### Robustness
Evaluate changes in:
- viewpoint
- illumination
- occlusion
- object scale
- background complexity
- camera motion

The strongest scientific comparison is to evaluate Qwen3-VL, Qwen2.5-VL, YOLOE, OWLv2, GroundingDINO and specialized video/action models using the **same data, prompts, classes and metrics**.


In [ ]:
# Release resources before loading the next VLM experiment.

del model
del processor

clear_memory()

print("Qwen3-VL resources released.")
